# Avaliação de Qualidade de Dados — `voebem.silver.aerodromos`

Avaliação sistemática da tabela Silver do cadastro de aeródromos públicos da ANAC, cobrindo seis dimensões de qualidade de dados (DQ): **completude, unicidade, consistência de domínios, padronização textual, integridade referencial e conformidade de formato**. Ao final, são apresentados os tratamentos recomendados.

In [0]:
# ===================================================================
# DQ 1 — COMPLETUDE: contagem de nulos e vazios por coluna
# ===================================================================
from pyspark.sql.functions import col, count, when, lit, sum as _sum

tbl = "voebem.silver.aerodromos"
df = spark.table(tbl)
total_rows = df.count()

cols_dq = [
    "icao", "ciad", "nome", "municipio", "uf_nome",
    "municipio_servido", "uf_servido_nome", "latitude_dms",
    "longitude_dms", "altitude_m", "situacao"
]

null_counts = df.select([
    count(when(col(c).isNull() | (trim(col(c)) == ""), c)).alias(c)
    for c in cols_dq
]).collect()[0]

rows = []
for c in cols_dq:
    n = null_counts[c]
    pct = round((n / total_rows) * 100, 2) if total_rows else 0
    flag = "\u26a0\ufe0f CR\u00cdTICO" if pct > 5 else ("\u26a0\ufe0f ATEN\u00c7\u00c3O" if pct > 0 else "\u2705 OK")
    rows.append((c, n, pct, flag))

completude_df = spark.createDataFrame(rows, ["coluna", "nulos", "pct_nulos", "status"])
print(f"Total de registros: {total_rows}")
display(completude_df.orderBy(col("pct_nulos").desc()))

In [0]:
# ===================================================================
# DQ 2 — UNICIDADE: verificar duplicidades em icao e ciad
# ===================================================================
from pyspark.sql.functions import col, count as _count, trim

df = spark.table("voebem.silver.aerodromos")

# Duplicidades por icao (chave primária esperada)
dups_icao = (df.groupBy("icao")
             .agg(_count("*").alias("qt"))
             .filter(col("qt") > 1)
             .orderBy(col("qt").desc()))
n_dups_icao = dups_icao.count()

# Duplicidades por ciad
dups_ciad = (df.groupBy("ciad")
             .agg(_count("*").alias("qt"))
             .filter(col("qt") > 1)
             .orderBy(col("qt").desc()))
n_dups_ciad = dups_ciad.count()

# icao nulo
n_icao_null = df.filter(col("icao").isNull() | (trim(col("icao")) == "")).count()

print(f"Duplicidades em icao: {n_dups_icao}")
print(f"Duplicidades em ciad: {n_dups_ciad}")
print(f"icao nulo/vazio: {n_icao_null}")

if n_dups_icao > 0:
    print("\nRegistros duplicados por icao:")
    display(dups_icao.limit(20))

if n_dups_ciad > 0:
    print("\nRegistros duplicados por ciad:")
    display(dups_ciad.limit(20))

if n_dups_icao == 0 and n_dups_ciad == 0:
    print("\u2705 Nenhuma duplicidade encontrada em icao ou ciad.")

In [0]:
# ===================================================================
# DQ 3 — CONSISTÊNCIA DE DOMÍNIOS: situacao, uf_nome, formato DMS,
# padrões de icao/ciad, altitude
# ===================================================================
from pyspark.sql.functions import col, count, when, trim, upper, lower, regexp_extract, length

df = spark.table("voebem.silver.aerodromos")

# 3.1 — Domínio de situacao
dist_situacao = df.groupBy("situacao").agg(count("*").alias("qt")).orderBy(col("qt").desc())
print("=== Domínio de situacao ===")
display(dist_situacao)

# 3.2 — Domínio de uf_nome (deve cobrir 27 UFs brasileiras)
dist_uf = df.groupBy("uf_nome").agg(count("*").alias("qt")).orderBy(col("qt").desc())
print("\n=== Domínio de uf_nome ===")
display(dist_uf)

# 3.3 — Validar formato DMS de latitude (padrão esperado: NN°NN'NN"N ou S)
lat_invalida = df.filter(
    col("latitude_dms").isNull() |
    (col("latitude_dms") == "") |
    ~col("latitude_dms").rlike(r"^\d{1,2}\u00b0\d{2}'\d{2}\"[NS]$")
)
print(f"\nLatitude DMS fora do padrão: {lat_invalida.count()}")
if lat_invalida.count() > 0:
    display(lat_invalida.select("icao", "nome", "latitude_dms").limit(20))

# 3.4 — Validar formato DMS de longitude (padrão esperado: NNN°NN'NN"E ou W)
lon_invalida = df.filter(
    col("longitude_dms").isNull() |
    (col("longitude_dms") == "") |
    ~col("longitude_dms").rlike(r"^\d{1,3}\u00b0\d{2}'\d{2}\"[EW]$")
)
print(f"Longitude DMS fora do padrão: {lon_invalida.count()}")
if lon_invalida.count() > 0:
    display(lon_invalida.select("icao", "nome", "longitude_dms").limit(20))

# 3.5 — Validar padrão ICAO (4 letras maiúsculas)
icao_invalido = df.filter(
    col("icao").isNull() |
    (col("icao") == "") |
    ~col("icao").rlike(r"^[A-Z]{4}$")
)
print(f"\nICAO fora do padrão (4 letras maiúsculas): {icao_invalido.count()}")
if icao_invalido.count() > 0:
    display(icao_invalido.select("icao", "nome").limit(20))

# 3.6 — Validar padrão CIAD (2 letras UF + 4 dígitos)
ciad_invalido = df.filter(
    col("ciad").isNull() |
    (col("ciad") == "") |
    ~col("ciad").rlike(r"^[A-Z]{2}\d{4}$")
)
print(f"CIAD fora do padrão (UF+dígitos): {ciad_invalido.count()}")
if ciad_invalido.count() > 0:
    display(ciad_invalido.select("ciad", "nome").limit(20))

# 3.7 — Verificar altitude: valores negativos ou extremos
alt_suspeitos = df.filter(
    col("altitude_m").isNull() |
    (col("altitude_m") < 0) |
    (col("altitude_m") > 6000)  # ponto mais alto do Brasil ~2900m
)
print(f"\nAltitude suspeita (< 0 ou > 6000m): {alt_suspeitos.count()}")
if alt_suspeitos.count() > 0:
    display(alt_suspeitos.select("icao", "nome", "altitude_m").limit(20))

In [0]:
# ===================================================================
# DQ 4 — PADRONIZAÇÃO TEXTUAL: verificar caixa (UPPER vs Title),
# espaços no início/fim, e possíveis inconsistências de acentuação
# ===================================================================
from pyspark.sql.functions import col, count, when, trim, upper, lower, length, regexp_replace

df = spark.table("voebem.silver.aerodromos")

colunas_texto = ["nome", "municipio", "uf_nome", "municipio_servido", "uf_servido_nome"]

resultados = []
for c in colunas_texto:
    total = df.filter(col(c).isNotNull()).count()
    # Está em UPPER (todas as letras maiúsculas)?
    n_upper = df.filter((col(c).isNotNull()) & (col(c) == upper(col(c)))).count()
    # Está em lower (todas minúsculas)?
    n_lower = df.filter((col(c).isNotNull()) & (col(c) == lower(col(c)))).count()
    # Tem espaços no início ou fim?
    n_espacos = df.filter((col(c).isNotNull()) & (col(c) != trim(col(c)))).count()
    # Nome com apenas uma palavra (possível abreviação)
    n_palavra_unica = df.filter((col(c).isNotNull()) & (~col(c).contains(" "))).count()
    resultados.append((c, total, n_upper, n_lower, n_espacos, n_palavra_unica))

pad_df = spark.createDataFrame(resultados, [
    "coluna", "total", "em_UPPER", "em_lower", "com_espacos_borda", "uma_palavra"
])
print("=== Análise de padronização textual ===")
print("'em_UPPER' = valor é todo em maiúsculas (provável inconsistência se misturado)")
display(pad_df)

# Verificar mistura de caixa em 'nome'
nome_upper = df.filter((col("nome").isNotNull()) & (col("nome") == upper(col("nome")))).count()
nome_title = df.filter((col("nome").isNotNull()) & (col("nome") != upper(col("nome")))).count()
print(f"\nColuna 'nome': {nome_upper} em UPPER vs {nome_title} em Title/Mixed")

# Exemplos de nomes em UPPER
if nome_upper > 0:
    print("\nExemplos de nomes em UPPER:")
    display(df.filter((col("nome").isNotNull()) & (col("nome") == upper(col("nome")))).select("icao", "nome").limit(10))

# Exemplos de nomes em Title
if nome_title > 0:
    print("\nExemplos de nomes em Title/Mixed:")
    display(df.filter((col("nome").isNotNull()) & (col("nome") != upper(col("nome")))).select("icao", "nome").limit(10))

In [0]:
# ===================================================================
# DQ 5 — INTEGRIDADE REFERENCIAL: verificar join com voebem.silver.vra
# Quantos icao_origem/icao_destino do VRA não existem no cadastro de aerodromos?
# ===================================================================
from pyspark.sql.functions import col, count, broadcast

aero = spark.table("voebem.silver.aerodromos")
vra = spark.table("voebem.silver.vra")

# Conjunto de icaos válidos no cadastro
aero_icaos = aero.select(col("icao").alias("icao_aero")).distinct()

# icaos únicos usados como origem no VRA
origem_icaos = vra.select(col("icao_origem").alias("icao")).distinct().filter(col("icao").isNotNull())

# icaos únicos usados como destino no VRA
destino_icaos = vra.select(col("icao_destino").alias("icao")).distinct().filter(col("icao").isNotNull())

# Origens não cadastradas
origem_nao_cad = (origem_icaos.join(aero_icaos, origem_icaos["icao"] == aero_icaos["icao_aero"], "left_anti"))
n_origem_nao_cad = origem_nao_cad.count()

# Destinos não cadastrados
destino_nao_cad = (destino_icaos.join(aero_icaos, destino_icaos["icao"] == aero_icaos["icao_aero"], "left_anti"))
n_destino_nao_cad = destino_nao_cad.count()

n_origem_total = origem_icaos.count()
n_destino_total = destino_icaos.count()

print(f"Total de icaos únicos como origem no VRA: {n_origem_total}")
print(f"Total de icaos únicos como destino no VRA: {n_destino_total}")
print(f"Origens não cadastradas no aerodromos: {n_origem_nao_cad} ({round(n_origem_nao_cad/n_origem_total*100,1) if n_origem_total else 0}%)")
print(f"Destinos não cadastrados no aerodromos: {n_destino_nao_cad} ({round(n_destino_nao_cad/n_destino_total*100,1) if n_destino_total else 0}%)")

if n_origem_nao_cad > 0:
    print("\nExemplos de origens não cadastradas (prováveis aeródromos estrangeiros):")
    display(origem_nao_cad.orderBy("icao").limit(20))

if n_destino_nao_cad > 0:
    print("\nExemplos de destinos não cadastrados:")
    display(destino_nao_cad.orderBy("icao").limit(20))

In [0]:
# ===================================================================
# DQ 6 — CONSISTÊNCIA ENTRE UF E MUNICÍPIO: verificar se
# uf_nome == uf_servido_nome e municipio == municipio_servido
# quando deveriam ser coerentes
# ===================================================================
from pyspark.sql.functions import col, count, when

df = spark.table("voebem.silver.aerodromos")

total = df.count()

# Casos onde uf_nome difere de uf_servido_nome
uf_diff = df.filter(
    col("uf_nome").isNotNull() &
    col("uf_servido_nome").isNotNull() &
    (col("uf_nome") != col("uf_servido_nome"))
)
print(f"Registros onde uf_nome \u2260 uf_servido_nome: {uf_diff.count()} de {total}")
if uf_diff.count() > 0:
    display(uf_diff.select("icao", "nome", "municipio", "uf_nome", "municipio_servido", "uf_servido_nome").limit(20))

# Casos onde municipio difere de municipio_servido (esperado em alguns casos, mas vale documentar)
mun_diff = df.filter(
    col("municipio").isNotNull() &
    col("municipio_servido").isNotNull() &
    (col("municipio") != col("municipio_servido"))
)
print(f"\nRegistros onde municipio \u2260 municipio_servido: {mun_diff.count()} de {total}")
if mun_diff.count() > 0:
    display(mun_diff.select("icao", "nome", "municipio", "municipio_servido").limit(20))

# Casos onde municipio IS NULL mas municipio_servido IS NOT NULL
mun_null_servido_ok = df.filter(
    col("municipio").isNull() &
    col("municipio_servido").isNotNull()
)
print(f"\nRegistros onde municipio IS NULL mas municipio_servido IS NOT NULL: {mun_null_servido_ok.count()}")
if mun_null_servido_ok.count() > 0:
    display(mun_null_servido_ok.select("icao", "nome", "municipio", "uf_nome", "municipio_servido", "uf_servido_nome").limit(20))

# Resumo da Avaliação e Tratamentos Recomendados

## Panorama Geral

| Métrica | Valor |
|---|---|
| Total de registros | **496** |
| Colunas avaliadas | 13 (11 de negócio + 2 de auditoria) |
| Dimensões DQ avaliadas | 6 |

---

## 1. Completude

| Coluna | Nulos | % | Status |
|---|---|---|---|
| `municipio` | 5 | 1,01% | \u26a0\ufe0f ATENÇÃO |
| `uf_nome` | 5 | 1,01% | \u26a0\ufe0f ATENÇÃO |
| `municipio_servido` | 2 | 0,40% | \u26a0\ufe0f ATENÇÃO |
| `uf_servido_nome` | 2 | 0,40% | \u26a0\ufe0f ATENÇÃO |
| `icao`, `ciad`, `nome`, `latitude_dms`, `longitude_dms`, `altitude_m`, `situacao` | 0 | 0% | \u2705 OK |

**Tratamento recomendado:**
* Investigar os **5 registros** com `municipio` e `uf_nome` nulos — são aeródromos onde o município de localização física não foi publicado pela ANAC (ex.: SBIZ — Prefeito Renato Moreira, em Imperatriz/MA). Avaliar se é possível enriquecer com `municipio_servido`/`uf_servido_nome` quando estes não forem nulos.
* Documentar como **propriedade da fonte** (ANAC não publica) e não como defeito de pipeline.

---

## 2. Unicidade

| Verificação | Resultado |
|---|---|
| Duplicidades em `icao` | **0** \u2705 |
| Duplicidades em `ciad` | **0** \u2705 |
| `icao` nulo/vazio | **0** \u2705 |

**Conclusão:** `icao` é chave primária natural, sem duplicidades. Nenhum tratamento necessário.

---

## 3. Consistência de Domínios

| Verificação | Resultado |
|---|---|
| Domínio de `situacao` | 2 valores: `Cadastrado` (465) e `Interditado` (31) \u2705 |
| Domínio de `uf_nome` | 27 UFs + 5 nulos \u2705 |
| Formato DMS de `latitude_dms` | 0 fora do padrão \u2705 |
| Formato DMS de `longitude_dms` | 0 fora do padrão \u2705 |
| Padrão ICAO (4 letras) | **11 fora do padrão** \u26a0\ufe0f |
| Padrão CIAD (UF+4 dígitos) | 0 fora do padrão \u2705 |
| Altitude (< 0 ou > 6000m) | 0 suspeitos \u2705 |

**Tratamento recomendado:**
* Os **11 códigos ICAO com dígitos** (ex.: `SN6L`, `SJ4Y`, `SDH2`) são válidos no cadastro da ANAC para aeródromos menores, mas **violam a convenção ICAO internacional** (4 letras). Recomenda-se: (a) documentar essa exceção nos comentários da coluna, e (b) se houver consumo externo que exige ICAO puro, adicionar uma flag booleana `icao_nao_conforme`.

---

## 4. Padronização Textual

| Coluna | Total | Em UPPER | Em Title | Espaços borda |
|---|---|---|---|---|
| `nome` | 496 | 19 (3,8%) | 477 (96,2%) | 0 |
| `municipio` | 491 | **491 (100%)** | 0 | 0 |
| `uf_nome` | 491 | 0 | 491 (100%) | 0 |
| `municipio_servido` | 494 | 0 | 494 (100%) | 0 |
| `uf_servido_nome` | 494 | 0 | 494 (100%) | 0 |

**Tratamento recomendado:**
* **`municipio`** está 100% em UPPER, enquanto `municipio_servido` está em Title Case — **inconsistência de padronização**. Recomenda-se aplicar `initcap()` em `municipio` para alinhar com `municipio_servido`.
* **`nome`**: 19 registros em UPPER misturados com 477 em Title Case. Aplicar `initcap()` para uniformizar, preservando siglas conhecidas (ex.: siglas IATA, nomes próprios).

---

## 5. Integridade Referencial (join com VRA)

| Verificação | Resultado |
|---|---|
| ICAOs únicos como origem no VRA | 512 |
| ICAOs únicos como destino no VRA | 519 |
| Origens sem cadastro em `aerodromos` | **320 (62,5%)** \u26a0\ufe0f |
| Destinos sem cadastro em `aerodromos` | **327 (63,0%)** \u26a0\ufe0f |

**Tratamento recomendado:**
* A **maioria dos ICAOs não cadastrados são aeródromos estrangeiros** (ex.: `BIKF`-Islândia, `CYYZ`-Canadá, `DAAG`-Argélia). Isso é **propriedade da fonte** (ANAC só cadastra aeródromos brasileiros), não defeito.
* Recomenda-se: (a) documentar essa limitação no comentário da tabela, (b) considerar criar uma tabela complementar `silver.aerodromos_estrangeiros` com nomes/localização dos aeroportos internacionais extraídos do VRA, ou enriquecer via API externa.

---

## 6. Consistência entre Colunas Relacionadas

| Verificação | Resultado |
|---|---|
| `uf_nome` \u2260 `uf_servido_nome` | 0 de 496 \u2705 |
| `municipio` \u2260 `municipio_servido` | **489 de 496 (98,6%)** \u26a0\ufe0f |
| `municipio` IS NULL mas `municipio_servido` IS NOT NULL | **5** \u26a0\ufe0f |

**Tratamento recomendado:**
* A diferença em 98,6% é **falsa positiva**: `municipio` está em UPPER e `municipio_servido` em Title Case, então a comparação string falha mesmo quando o município é o mesmo. **Após aplicar `initcap()` em `municipio`** (recomendado no item 4), re-rodar esta verificação para identificar divergências reais.
* Os **5 registros** com `municipio` nulo mas `municipio_servido` preenchido podem ser enriquecidos copiando o valor de `municipio_servido` se a regra de negócio permitir.

---

## Plano de Tratamento Priorizado

| Prioridade | Dimensão DQ | Ação | Impacto |
|---|---|---|---|
| **Alta** | Padronização | Aplicar `initcap()` em `municipio` e `nome` | Uniformiza caixa textual em toda a tabela |
| **Alta** | Completude | Enriquecer 5 registros com `municipio` nulo usando `municipio_servido` | Reduz nulos de 5 para 0 |
| **Média** | Consistência | Documentar 11 ICAOs com dígitos como exceção válida da ANAC | Evita falsos positivos em validações futuras |
| **Média** | Integridade | Documentar que aeródromos estrangeiros não constam (propriedade da fonte) | Evita investigações desnecessárias |
| **Baixa** | Conformidade | Adicionar colunas derivadas `latitude_decimal` e `longitude_decimal` (converter DMS \u2192 graus decimais) | Facilita análises geoespaciais |
| **Baixa** | Governança | Adicionar `COMMENT` na tabela documentando limitações e cobrança | Rastreabilidade e transparência |

In [0]:
# ===================================================================
# DQ 7 — VALIDAÇÕES DE CONTEXTO DE VALORES E NEGÓCIO
# Regras de negócio sobre os dados de aeródromos, indo além das
# verificações estruturais (formato, nulidade, unicidade) e
# verificando coerência semântica e regras do domínio ANAC.
# ===================================================================
from pyspark.sql.functions import (
    col, count, when, trim, upper, lower, regexp_extract,
    initcap, lit, concat, expr, min as _min, max as _max, collect_set
)

df = spark.table("voebem.silver.aerodromos")
vra = spark.table("voebem.silver.vra")
total = df.count()

problemas = []

# -----------------------------------------------------------------
# 7.1 — Latitude/Longitude dentro do território brasileiro
#   Brasil: lat  -33,75 a +5,38   → DMS de 0° a 34° S
#           lon  -73,99 a -34,79  → DMS de 34° a 74° W
# -----------------------------------------------------------------
# Extrair componentes DMS para graus decimais
lat_dec = expr("""
    CAST(SUBSTRING(latitude_dms, 1, LOCATE('°', latitude_dms)-1) AS DOUBLE)
  + CAST(SUBSTRING(latitude_dms, LOCATE('°', latitude_dms)+1,
        LOCATE("'", latitude_dms)-LOCATE('°', latitude_dms)-1) AS DOUBLE)/60
  + CAST(SUBSTRING(latitude_dms, LOCATE("'", latitude_dms)+1,
        LOCATE('"', latitude_dms)-LOCATE("'", latitude_dms)-1) AS DOUBLE)/3600
""")
lon_dec = expr("""
    CAST(SUBSTRING(longitude_dms, 1, LOCATE('°', longitude_dms)-1) AS DOUBLE)
  + CAST(SUBSTRING(longitude_dms, LOCATE('°', longitude_dms)+1,
        LOCATE("'", longitude_dms)-LOCATE('°', longitude_dms)-1) AS DOUBLE)/60
  + CAST(SUBSTRING(longitude_dms, LOCATE("'", longitude_dms)+1,
        LOCATE('"', longitude_dms)-LOCATE("'", longitude_dms)-1) AS DOUBLE)/3600
""")

df_geo = df.withColumn("lat_decimal", lat_dec) \
           .withColumn("lon_decimal", lon_dec)

# Ajustar sinal: Sul e Oeste são negativos
df_geo = df_geo.withColumn(
    "lat_decimal",
    when(col("latitude_dms").contains("S"), -col("lat_decimal"))
    .otherwise(col("lat_decimal"))
).withColumn(
    "lon_decimal",
    when(col("longitude_dms").contains("W"), -col("lon_decimal"))
    .otherwise(col("lon_decimal"))
)

fora_brasil = df_geo.filter(
    (col("lat_decimal") > 5.4) | (col("lat_decimal") < -33.75) |
    (col("lon_decimal") > -34.0) | (col("lon_decimal") < -74.0)
)
n_fora = fora_brasil.count()
print(f"=== 7.1 — Coordenadas fora do território brasileiro ===")
print(f"Aeródromos com lat/lon fora dos limites do Brasil: {n_fora} de {total}")
if n_fora > 0:
    display(fora_brasil.select("icao", "nome", "municipio", "uf_nome",
                               "latitude_dms", "longitude_dms",
                               "lat_decimal", "lon_decimal").limit(20))
problemas.append(("7.1 Coordenadas fora do Brasil", n_fora))

# -----------------------------------------------------------------
# 7.2 — Prefixo do CIAD (2 primeiros caracteres) coerente com uf_nome
#   Ex.: CIAD "SP0001" → uf_nome deve conter "São Paulo"
# -----------------------------------------------------------------
uf_map = {
    "AC": "ACRE", "AL": "ALAGOAS", "AP": "AMAPÁ", "AM": "AMAZONAS",
    "BA": "BAHIA", "CE": "CEARÁ", "DF": "DISTRITO FEDERAL",
    "ES": "ESPÍRITO SANTO", "GO": "GOIÁS", "MA": "MARANHÃO",
    "MT": "MATO GROSSO", "MS": "MATO GROSSO DO SUL", "MG": "MINAS GERAIS",
    "PA": "PARÁ", "PB": "PARAÍBA", "PR": "PARANÁ", "PE": "PERNAMBUCO",
    "PI": "PIAUÍ", "RJ": "RIO DE JANEIRO", "RN": "RIO GRANDE DO NORTE",
    "RS": "RIO GRANDE DO SUL", "RO": "RONDÔNIA", "RR": "RORAIMA",
    "SC": "SANTA CATARINA", "SE": "SERGIPE", "SP": "SÃO PAULO", "TO": "TOCANTINS"
}

df_ciad = df.withColumn("ciad_uf", upper(regexp_extract(col("ciad"), r"^([A-Z]{2})", 1)))
rows_ciad = []
for uf_sigla, uf_nome_completo in uf_map.items():
    rows_ciad.append((uf_sigla, uf_nome_completo))
uf_ref = spark.createDataFrame(rows_ciad, ["uf_sigla", "uf_nome_ref"])

df_ciad = df_ciad.join(uf_ref, df_ciad["ciad_uf"] == uf_ref["uf_sigla"], "left")

ciad_incoerente = df_ciad.filter(
    col("uf_nome").isNotNull() &
    col("uf_nome_ref").isNotNull() &
    (upper(trim(col("uf_nome"))) != upper(trim(col("uf_nome_ref"))))
)
n_ciad_inc = ciad_incoerente.count()
print(f"\n=== 7.2 — Prefixo CIAD coerente com uf_nome ===")
print(f"Registros onde prefixo do CIAD não corresponde à UF: {n_ciad_inc} de {total}")
if n_ciad_inc > 0:
    display(ciad_incoerente.select("icao", "ciad", "ciad_uf", "uf_nome", "uf_nome_ref", "nome").limit(20))
problemas.append(("7.2 Prefixo CIAD vs UF", n_ciad_inc))

# -----------------------------------------------------------------
# 7.3 — Aeródromos com mesmo nome em municípios diferentes
#   Pode indicar erro de cadastro ou duplicidade semântica
# -----------------------------------------------------------------
nomes_duplicados = (df.filter(col("nome").isNotNull())
    .groupBy("nome")
    .agg(count("*").alias("qt"), collect_set("municipio").alias("municipios"))
    .filter((col("qt") > 1) & (expr("size(municipios)") > 1))
    .orderBy(col("qt").desc()))
n_nomes_dup = nomes_duplicados.count()
print(f"\n=== 7.3 — Mesmo nome em municípios diferentes ===")
print(f"Nomes de aeródromo compartilhados entre municípios distintos: {n_nomes_dup}")
if n_nomes_dup > 0:
    display(nomes_duplicados.select("nome", "qt", "municipios").limit(20))
problemas.append(("7.3 Mesmo nome / municípios diferentes", n_nomes_dup))

# -----------------------------------------------------------------
# 7.4 — Aeródromos Interditados que aparecem como ativos no VRA
#   Um aeródromo Interditado não deveria ter movimentações recentes
# -----------------------------------------------------------------
aero_interditado = df.filter(col("situacao") == "Interditado") \
    .select(col("icao").alias("icao_int"))

vra_ativo = vra.select(
    col("icao_origem").alias("icao_int"), col("icao_destino"),
    col("partida_real"), col("chegada_real")
).filter(col("icao_origem").isNotNull())

interditado_com_vra = (aero_interditado
    .join(vra.filter(col("icao_origem") == aero_interditado["icao_int"]
                      ) if False else vra.select(col("icao_origem").alias("icao_int")),
          "icao_int", "inner")
    .distinct())

# Abordagem mais simples: contar interditados usados no VRA
icaos_interditados = [r.icao_int for r in aero_interditado.collect()]
n_interditados = len(icaos_interditados)

vra_origem_icaos = vra.select("icao_origem").distinct() \
    .filter(col("icao_origem").isin(icaos_interditados)) \
    .count()

vra_destino_icaos = vra.select("icao_destino").distinct() \
    .filter(col("icao_destino").isin(icaos_interditados)) \
    .count()

print(f"\n=== 7.4 — Aeródromos Interditados com movimentação no VRA ===")
print(f"Total de aeródromos Interditados: {n_interditados}")
print(f"Usados como origem no VRA: {vra_origem_icaos}")
print(f"Usados como destino no VRA: {vra_destino_icaos}")
problemas.append(("7.4 Interditados ativos no VRA", vra_origem_icaos + vra_destino_icaos))

# -----------------------------------------------------------------
# 7.5 — Altitude plausível para a UF do aeródromo
#   Cada UF tem faixas altimétricas conhecidas; flag extremos
# -----------------------------------------------------------------
alt_stats = df.filter(col("altitude_m").isNotNull()) \
    .groupBy("uf_nome") \
    .agg(
        _min("altitude_m").alias("alt_min"),
        _max("altitude_m").alias("alt_max"),
        expr("percentile(altitude_m, 0.5)").alias("alt_mediana"),
        count("*").alias("qt")
    ) \
    .orderBy(col("alt_max").desc())

print(f"\n=== 7.5 — Estatísticas de altitude por UF ===")
display(alt_stats)

# Identificar outliers: altitude > 3x mediana da UF ou < 10% da mediana
alt_with_median = df.join(
    alt_stats.select("uf_nome", col("alt_mediana").alias("med_uf")),
    "uf_nome", "left"
)

alt_outlier = alt_with_median.filter(
    col("altitude_m").isNotNull() &
    col("med_uf").isNotNull() &
    (
        (col("altitude_m") > col("med_uf") * 3) |
        (col("altitude_m") < col("med_uf") * 0.10)
    )
)
n_outlier = alt_outlier.count()
print(f"\nAeródromos com altitude discrepante (>3x ou <10% da mediana da UF): {n_outlier}")
if n_outlier > 0:
    display(alt_outlier.select("icao", "nome", "uf_nome", "altitude_m", "med_uf").limit(20))
problemas.append(("7.5 Altitude discrepante por UF", n_outlier))

# -----------------------------------------------------------------
# 7.6 — Situação vs existência de operações: aeródromos Cadastrados
#   que nunca aparecem no VRA (possíveis aeródromos sem operação regular)
# -----------------------------------------------------------------
aero_cadastrado = df.filter(col("situacao") == "Cadastrado") \
    .select(col("icao").alias("icao_cad"))

vra_icaos = vra.select(
    col("icao_origem").alias("icao_cad"),
    col("icao_destino")
).select("icao_cad").union(
    vra.select(col("icao_destino").alias("icao_cad"))
).filter(col("icao_cad").isNotNull()).distinct()

cadastrado_sem_vra = aero_cadastrado.join(vra_icaos, "icao_cad", "left_anti")
n_sem_vra = cadastrado_sem_vra.count()
n_cad_total = aero_cadastrado.count()
print(f"\n=== 7.6 — Cadastrados sem nenhuma movimentação no VRA ===")
print(f"Aeródromos Cadastrados: {n_cad_total}")
print(f"Nunca apareceram no VRA (origem ou destino): {n_sem_vra} "
      f"({round(n_sem_vra/n_cad_total*100,1) if n_cad_total else 0}%)")
if n_sem_vra > 0:
    display(cadastrado_sem_vra.join(df, cadastrado_sem_vra["icao_cad"] == df["icao"], "left")
            .select("icao", "nome", "municipio", "uf_nome", "situacao").limit(20))
problemas.append(("7.6 Cadastrados sem operação no VRA", n_sem_vra))

# -----------------------------------------------------------------
# 7.7 — Múltiplos aeródromos no mesmo município (concentração)
#   Informações de densidade: municípios com muitos aeródromos
# -----------------------------------------------------------------
por_mun = (df.filter(col("municipio").isNotNull())
    .groupBy("municipio", "uf_nome")
    .agg(count("*").alias("qt_aerodromos"))
    .orderBy(col("qt_aerodromos").desc()))

mun_multi = por_mun.filter(col("qt_aerodromos") > 3)
print(f"\n=== 7.7 — Municípios com mais de 3 aeródromos ===")
print(f"Total de municípios com >3 aeródromos: {mun_multi.count()}")
display(mun_multi.limit(20))
problemas.append(("7.7 Municípios com >3 aeródromos", mun_multi.count()))

# -----------------------------------------------------------------
# Resumo consolidado
# -----------------------------------------------------------------
print("\n" + "="*70)
print("RESUMO — VALIDAÇÕES DE CONTEXTO DE NEGÓCIO")
print("="*70)
resumo = spark.createDataFrame(problemas, ["validacao", "registros_afetados"])
display(resumo.orderBy(col("registros_afetados").desc()))

In [0]:
# ===================================================================
# DQ GATE — Impedir carga de registros com icao vazio/nulo na silver
# ===================================================================
# Estratégia em duas camadas:
#   1) Filtro ativo antes do MERGE/INSERT  → rejeita e registra
#   2) Constraint Delta na tabela silver    → barreira definitiva
# ===================================================================
from pyspark.sql.functions import col, trim, lit, current_timestamp

TABELA_SILVER = "voebem.silver.aerodromos"
TABELA_REJEITADOS = "voebem.silver.aerodromos_dq_rejeitados"

# -----------------------------------------------------------------
# 1) Garantir constraint na tabela silver (executar uma vez)
#    Impede INSERT/MERGE de qualquer linha com icao nulo ou vazio.
# -----------------------------------------------------------------
spark.sql(f"""
ALTER TABLE {TABELA_SILVER} ADD CONSTRAINT IF NOT EXISTS icao_nao_vazio
    CHECK (icao IS NOT NULL AND trim(icao) != '')
""")
print(f"Constraint 'icao_nao_vazio' garantida em {TABELA_SILVER}")

# -----------------------------------------------------------------
# 2) Tabela de rejeitados (para auditoria e reprocessamento)
# -----------------------------------------------------------------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABELA_REJEITADOS} (
    icao        STRING,
    ciad        STRING,
    nome        STRING,
    motivo      STRING,
    rejeitado_em TIMESTAMP
) USING DELTA
""")
print(f"Tabela de rejeitados: {TABELA_REJEITADOS}")

# -----------------------------------------------------------------
# 3) Função reutilizável para filtrar df de entrada antes da carga
#    Uso:  df_valido, df_rejeitado = dq_gate_icao(df_entrada)
#          ... MERGE df_valido INTO silver ...
#          gravar df_rejeitado na tabela de auditoria
# -----------------------------------------------------------------
def dq_gate_icao(df, coluna_icao="icao"):
    """
    Separa registros válidos dos rejeitados (icao nulo ou vazio).
    Retorna (df_valido, df_rejeitado).
    """
    is_invalid = col(coluna_icao).isNull() | (trim(col(coluna_icao)) == lit(""))

    df_valido = df.filter(~is_invalid)

    df_rejeitado = (
        df.filter(is_invalid)
          .withColumn("motivo", lit("icao nulo ou vazio"))
          .withColumn("rejeitado_em", current_timestamp())
          .select(
              col(coluna_icao).alias("icao"),
              col("ciad") if "ciad" in df.columns else lit(None).alias("ciad"),
              col("nome") if "nome" in df.columns else lit(None).alias("nome"),
              "motivo", "rejeitado_em",
          )
    )
    return df_valido, df_rejeitado


# -----------------------------------------------------------------
# 4) Exemplo de uso no fluxo de carga bronze → silver
# ----------------------------------------------------------------==
# df_bronze = spark.table("voebem.bronze.aerodromos")
# df_valido, df_rejeitado = dq_gate_icao(df_bronze)
#
# # Gravar rejeitados para auditoria
# if df_rejeitado.count() > 0:
#     print(f"{df_rejeitado.count()} registro(s) rejeitado(s) por icao vazio")
#     df_rejeitado.write \
#         .format("delta") \
#         .mode("append") \
#         .saveAsTable(TABELA_REJEITADOS)
#
# # MERGE apenas dos válidos na silver
# # (a constraint Delta ainda atua como barreira de segurança final)
# from delta.tables import DeltaTable
# delta_tbl = DeltaTable.forName(spark, TABELA_SILVER)
# (delta_tbl.alias("t")
#     .merge(df_valido.alias("s"), "t.icao = s.icao")
#     .whenMatchedUpdateAll()
#     .whenNotMatchedInsertAll()
#     .execute())
# print("Carga concluída — apenas registros com icao válido na silver.")

# Demonstração rápida: aplicar o gate a um DataFrame existente
print("\n=== Demonstração do DQ Gate na tabela silver atual ===")
df_atual = spark.table(TABELA_SILVER)
df_valido, df_rejeitado = dq_gate_icao(df_atual)
n_total = df_atual.count()
n_valido = df_valido.count()
n_rejeitado = n_total - n_valido
print(f"Total de registros:    {n_total}")
print(f"Válidos (icao ok):    {n_valido}")
print(f"Rejeitados (icao vazio/nulo): {n_rejeitado}")
if n_rejeitado > 0:
    print("\nRegistros rejeitados:")
    display(df_rejeitado.limit(20))
else:
    print("\nNenhum registro rejeitado — todos os icao estão preenchidos. ✅")